In [ ]:
import pandas as pd
file_path = "/Users/sunjaelee/Library/CloudStorage/Dropbox/비교과/학회/heart_failure_clinical_records_dataset.csv"
data = pd.read_csv(file_path)

In [ ]:
pd.set_option('display.float_format', '{:.10f}'.format)

In [ ]:
from scipy.stats import mannwhitneyu

results = []

group_0 = data[data['DEATH_EVENT'] == 0]
group_1 = data[data['DEATH_EVENT'] == 1]

columns = ["serum_creatinine", "ejection_fraction", "age", "creatinine_phosphokinase", "platelets", "serum_sodium"]

for column in columns:
    if column != 'DEATH_EVENT':  # 비교할 변수 제외
        stat, p_value = mannwhitneyu(group_0[column], group_1[column], alternative='two-sided')
        results.append({
            'Variable': column,
            'Mann-Whitney U Statistic': stat,
            'P-value': p_value
        })

# 결과를 데이터프레임으로 정리
results_df = pd.DataFrame(results)



In [ ]:
results_df

In [ ]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# 데이터 준비
# 필요한 열만 선택 (time, DEATH_EVENT, 및 설명 변수들)
columns_to_use = [
    'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'high_blood_pressure', 'platelets',
    'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT'
]
prepared_data = data[columns_to_use]

# Cox 모델 적합
cox_model = CoxPHFitter()
cox_model.fit(prepared_data, duration_col='time', event_col='DEATH_EVENT')

# 모델 요약 출력
cox_summary = cox_model.summary

# Concordance Index 계산
c_index = concordance_index(prepared_data['time'], -cox_model.predict_partial_hazard(prepared_data), prepared_data['DEATH_EVENT'])

cox_summary, c_index


In [ ]:
!pip install lifelines.plotting

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

# Forest Plot: 변수별 Hazard Ratio 시각화
cox_model.plot()
plt.title("Hazard Ratios (Forest Plot)")
plt.show()

# 비례 위험 가정 검토: Schoenfeld 잔차
cox_model.check_assumptions(prepared_data, p_value_threshold=0.05)